# Constraints

Constrained inference is conin's distinguishing feature. A constraint
restricts which state assignments are considered feasible during inference.
This notebook covers how to define constraints for every model type; the
[inference notebook](inference.ipynb) covers how to solve with them.

conin supports several constraint backends, each with its own syntax:

| Backend | Style | Key idea |
| --- | --- | --- |
| **Oracle** | Python predicate | `func(states) -> bool` |
| **Pyomo** | Imperative model mutation | `M.c = pe.Constraint(expr=...)` |
| **Toulbar2** | Imperative model mutation | `M.AddGeneralizedLinearConstraint(...)` |
| **Algebraic** | Declarative expression | `return sum(...) <= 5` |

To make the backends easy to compare, each section below implements
**one fixed constraint per model type** across all four backends:

| Model | Constraint |
| --- | --- |
| HMM | `"rainy"` must appear at least 3 times |
| BN | `Dyspnoea` and `Xray` cannot take the same value |
| MN | All three nodes must have different values |
| DBN | Variable `A` must have the same value at t=0 and t=1 |

In [ ]:
import pyomo.environ as pe

from conin.hidden_markov_model import HiddenMarkovModel, ConstrainedHiddenMarkovModel
from conin.bayesian_network import (
    DiscreteBayesianNetwork,
    DiscreteCPD,
    ConstrainedDiscreteBayesianNetwork,
)
from conin.markov_network import (
    DiscreteMarkovNetwork,
    DiscreteFactor,
    ConstrainedDiscreteMarkovNetwork,
)
from conin.dynamic_bayesian_network import (
    DynamicDiscreteBayesianNetwork,
    ConstrainedDynamicDiscreteBayesianNetwork,
)
from conin.constraints import (
    OracleConstraint,
    oracle_constraint_fn,
    pyomo_constraint_fn,
    toulbar2_constraint_fn,
    algebraic_constraint_fn,
)

## Setup

We build one model of each type. See the per-model basics notebooks
([HMM](HMM_basics.ipynb), [BN](BN_basics.ipynb), [MN](MN_basics.ipynb),
[DBN](DBN_basics.ipynb)) for construction details.

In [ ]:
# --- HMM: weather model ---
hmm = HiddenMarkovModel()
hmm.load_model(
    start_probs={"sunny": 0.6, "rainy": 0.4},
    transition_probs={
        ("sunny", "sunny"): 0.7, ("sunny", "rainy"): 0.3,
        ("rainy", "sunny"): 0.4, ("rainy", "rainy"): 0.6,
    },
    emission_probs={
        ("sunny", "walk"): 0.6, ("sunny", "shop"): 0.3, ("sunny", "clean"): 0.1,
        ("rainy", "walk"): 0.1, ("rainy", "shop"): 0.4, ("rainy", "clean"): 0.5,
    },
)

# --- BN: cancer diagnosis ---
bn = DiscreteBayesianNetwork(
    states={
        "Pollution": ["low", "high"], "Smoker": ["yes", "no"],
        "Cancer": ["yes", "no"], "Xray": ["positive", "negative"],
        "Dyspnoea": ["yes", "no"],
    },
    cpds=[
        DiscreteCPD(node="Pollution", values={"low": 0.9, "high": 0.1}),
        DiscreteCPD(node="Smoker", values={"yes": 0.3, "no": 0.7}),
        DiscreteCPD(node="Cancer", parents=["Smoker", "Pollution"], values={
            ("yes", "low"): {"yes": 0.03, "no": 0.97},
            ("yes", "high"): {"yes": 0.05, "no": 0.95},
            ("no", "low"): {"yes": 0.001, "no": 0.999},
            ("no", "high"): {"yes": 0.02, "no": 0.98},
        }),
        DiscreteCPD(node="Xray", parents=["Cancer"], values={
            "yes": {"positive": 0.9, "negative": 0.1},
            "no": {"positive": 0.2, "negative": 0.8},
        }),
        DiscreteCPD(node="Dyspnoea", parents=["Cancer"], values={
            "yes": {"yes": 0.65, "no": 0.35},
            "no": {"yes": 0.3, "no": 0.7},
        }),
    ],
)

# --- MN: three-node network ---
mn = DiscreteMarkovNetwork(
    states={"A": [0, 1, 2], "B": [0, 1, 2], "C": [0, 1, 2]},
    factors=[
        DiscreteFactor(nodes=["A"], values={0: 1, 1: 1, 2: 2}),
        DiscreteFactor(nodes=["B"], values={0: 1, 1: 1, 2: 3}),
        DiscreteFactor(nodes=["C"], values={0: 1, 1: 2, 2: 1}),
        DiscreteFactor(nodes=["A", "B"], values={(i, j): 1 for i in range(3) for j in range(3)}),
        DiscreteFactor(nodes=["B", "C"], values={(i, j): 1 for i in range(3) for j in range(3)}),
        DiscreteFactor(nodes=["A", "C"], values={(i, j): 1 for i in range(3) for j in range(3)}),
    ],
)

# --- DBN: two dynamic variables ---
dbn = DynamicDiscreteBayesianNetwork()
dbn.dynamic_states = {"A": [0, 1], "B": [0, 1]}
dbn.cpds = [
    DiscreteCPD(node=("A", 0), values=[0.9, 0.1]),
    DiscreteCPD(
        node=("B", dbn.t), parents=[("A", dbn.t)],
        values={0: [0.2, 0.8], 1: [0.9, 0.1]},
    ),
    DiscreteCPD(
        node=("A", dbn.t), parents=[("A", dbn.t - 1)],
        values={0: [0.2, 0.8], 1: [0.9, 0.1]},
    ),
]

print("HMM hidden states:", hmm.hidden_states)
print("BN nodes:         ", sorted(bn.states.keys()))
print("MN nodes:         ", sorted(mn.states.keys()))
print("DBN dynamic nodes:", dbn.dynamic_nodes)

## Oracle Constraints

An oracle constraint is a Python function that receives a **dict** of state
assignments and returns `True` if the assignment is feasible. The dict keys
depend on the model type:

| Model | Keys | Example |
| --- | --- | --- |
| HMM | time indices | `{0: "rainy", 1: "sunny"}` |
| BN / MN | node names | `{"Dyspnoea": "yes", "Xray": "positive"}` |
| DBN | `(node, time)` tuples | `{("A", 0): 0, ("A", 1): 1}` |

For **HMMs**, oracle constraints are used directly by A\* search — no
`nodes` parameter is needed. An optional `partial_func(T, states)` enables
early pruning of partial sequences.

For **BN, MN, and DBN**, the `nodes` parameter declares which variables the
predicate involves. At inference time, the constraint is materialized into a
factor by enumerating all assignments over the declared nodes.

In [ ]:
# HMM: "rainy" must appear at least 3 times
hmm_oracle = OracleConstraint(
    func=lambda states: sum(1 for v in states.values() if v == "rainy") >= 3,
    partial_func=lambda T, states: (
        sum(1 for v in states.values() if v == "rainy") + (T - len(states)) >= 3
    ),
)

# Test
print("HMM oracle:", hmm_oracle({0: "rainy", 1: "rainy", 2: "rainy", 3: "sunny"}))

In [ ]:
# BN: Dyspnoea and Xray must differ
@oracle_constraint_fn(nodes=["Dyspnoea", "Xray"])
def bn_oracle(states):
    return states["Dyspnoea"] != states["Xray"]


# MN: all three nodes must have different values
@oracle_constraint_fn(nodes=["A", "B", "C"])
def mn_oracle(states):
    return len(set(states.values())) == len(states)


# DBN: A must have the same value at t=0 and t=1
def dbn_oracle_nodes(data):
    for t in data.T:
        yield ("A", t)


@oracle_constraint_fn(nodes=dbn_oracle_nodes)
def dbn_oracle(states, data):
    return states[("A", 0)] == states[("A", 1)]


# Test the underlying predicates
print("BN oracle: ", bn_oracle.func({"Dyspnoea": "yes", "Xray": "positive"}))
print("MN oracle: ", mn_oracle.func({"A": 0, "B": 2, "C": 1}))

### Pre-built oracle constraint library

`conin.constraints.oracle` provides common HMM constraints so you don't
have to write them from scratch. These operate on the same dict format.

In [ ]:
from conin.constraints.oracle import (
    does_not_occur_constraint,
    has_minimum_number_of_occurences_constraint,
    has_maximum_number_of_occurences_constraint,
    appears_at_least_once_constraint,
    fix_final_state_constraint,
    and_constraints,
    or_constraints,
    not_constraint,
)

# The same HMM constraint using the library
hmm_oracle_lib = has_minimum_number_of_occurences_constraint(val="rainy", count=3)

test = {0: "rainy", 1: "rainy", 2: "sunny", 3: "rainy"}
print("rainy >= 3:", hmm_oracle_lib(test))

# Combinators
rainy_3_to_5 = and_constraints([
    has_minimum_number_of_occurences_constraint(val="rainy", count=3),
    has_maximum_number_of_occurences_constraint(val="rainy", count=5),
])
print("rainy 3-5: ", rainy_3_to_5(test))

## Pyomo Constraints

Pyomo constraints express restrictions as linear constraints over binary
decision variables. The decorated function receives a model `M` and
optionally a data object `D` (for time-indexed models like HMMs and DBNs).

`M.V()` returns a binary indicator variable:

| Model | Pattern | Meaning |
| --- | --- | --- |
| BN / MN | `M.V("Node", state)` | 1 if `Node == state` |
| HMM / DBN | `M.V("Node", t, state)` | 1 if `Node` at time `t` equals `state` |

NOTE: Pyomo constraints require a MIP solver (e.g., HiGHS) at inference
time.

In [ ]:
# HMM: "rainy" at least 3 times
@pyomo_constraint_fn()
def hmm_pyomo(M, D):
    M.rainy_lb = pe.Constraint(
        expr=sum(M.V("H", t, "rainy") for t in D.hmm.T) >= 3
    )


# BN: Dyspnoea and Xray must differ
@pyomo_constraint_fn()
def bn_pyomo(M):
    M.c = pe.ConstraintList()
    M.c.add(M.V("Dyspnoea", "yes") + M.V("Xray", "positive") <= 1)
    M.c.add(M.V("Dyspnoea", "no") + M.V("Xray", "negative") <= 1)

In [ ]:
# MN: all different — no two nodes share a state
@pyomo_constraint_fn()
def mn_pyomo(M):
    @M.Constraint([0, 1, 2])
    def diff(M, s):
        return M.V("A", s) + M.V("B", s) + M.V("C", s) <= 1


# DBN: A must be the same at t=0 and t=1
@pyomo_constraint_fn()
def dbn_pyomo(M, D):
    M.c = pe.ConstraintList()
    M.c.add(M.V("A", 0, 0) == M.V("A", 1, 0))

## Toulbar2 Constraints

Toulbar2 constraints target the Toulbar2 cost function network solver.
Constraints are added via `M.AddGeneralizedLinearConstraint(vars, op, rhs)`
where `op` is `"=="`, `">="`, or `"<="`. The same `M.V()` patterns apply.

NOTE: Requires the `pytoulbar2` package at inference time.

In [ ]:
# HMM: "rainy" at least 3 times
@toulbar2_constraint_fn()
def hmm_toulbar2(M, D):
    M.AddGeneralizedLinearConstraint(
        [M.V("H", t, "rainy") for t in D.hmm.T], ">=", 3
    )


# BN: Dyspnoea and Xray must differ
@toulbar2_constraint_fn()
def bn_toulbar2(M):
    M.AddGeneralizedLinearConstraint(
        [M.V("Dyspnoea", "yes"), M.V("Xray", "positive")], "<=", 1
    )
    M.AddGeneralizedLinearConstraint(
        [M.V("Dyspnoea", "no"), M.V("Xray", "negative")], "<=", 1
    )

In [ ]:
# MN: all different
@toulbar2_constraint_fn()
def mn_toulbar2(M):
    for s in [0, 1, 2]:
        M.AddGeneralizedLinearConstraint(
            [M.V("A", s), M.V("B", s), M.V("C", s)], "<=", 1
        )


# DBN: A must be the same at t=0 and t=1
@toulbar2_constraint_fn()
def dbn_toulbar2(M, D):
    M.AddGeneralizedLinearConstraint(
        [M.V("A", 0, 0), M.V("A", 1, 0, coef=-1)], "==", 0
    )

## Algebraic Constraints

Algebraic constraints use the same `M.V()` syntax but **return** symbolic
expressions instead of mutating the model. The backend automatically
translates these to either Pyomo or Toulbar2 format depending on the
inference method, making them the most portable option for linear
constraints.

NOTE: Requires the `smoek` package.

In [ ]:
# HMM: "rainy" at least 3 times
@algebraic_constraint_fn()
def hmm_algebraic(M, D):
    return sum(M.V("H", t, "rainy") for t in D.hmm.T) >= 3


# BN: Dyspnoea and Xray must differ
@algebraic_constraint_fn()
def bn_algebraic(M):
    return [
        M.V("Dyspnoea", "yes") + M.V("Xray", "positive") <= 1,
        M.V("Dyspnoea", "no") + M.V("Xray", "negative") <= 1,
    ]

In [ ]:
# MN: all different
@algebraic_constraint_fn()
def mn_algebraic(M):
    return [
        M.V("A", s) + M.V("B", s) + M.V("C", s) <= 1
        for s in [0, 1, 2]
    ]


# DBN: A must be the same at t=0 and t=1
@algebraic_constraint_fn()
def dbn_algebraic(M, D):
    return M.V("A", 0, 0) == M.V("A", 1, 0)

## Building Constrained Models

Each model type has a constrained wrapper that pairs a base model with a
list of constraints. All constraints in one model must use the same
backend.

In [ ]:
# HMM — requires initialize_chmm()
chmm = ConstrainedHiddenMarkovModel(hmm=hmm, constraints=[hmm_oracle])
chmm.initialize_chmm()
print("HMM  constraint type:", chmm.constraint_type)

# BN
cbn = ConstrainedDiscreteBayesianNetwork(bn, constraints=[bn_pyomo])
print("BN   # constraints:  ", len(cbn.constraints))

# MN
cmn = ConstrainedDiscreteMarkovNetwork(mn, constraints=[mn_pyomo])
print("MN   # constraints:  ", len(cmn.constraints))

# DBN
cdbn = ConstrainedDynamicDiscreteBayesianNetwork(dbn, constraints=[dbn_pyomo])
print("DBN  # constraints:  ", len(cdbn.constraints))

These constrained models can now be passed to `map_query` — see the
[inference notebook](inference.ipynb).

## Summary

### `M.V()` patterns

| Model | Oracle (dict keys) | Pyomo / Toulbar2 / Algebraic |
| --- | --- | --- |
| BN / MN | `states["Node"]` | `M.V("Node", state)` |
| HMM | `states[t]` | `M.V("H", t, state)` |
| DBN | `states[("Node", t)]` | `M.V("Node", t, state)` |

### Which constraint type works with which inference method?

| Constraint type | `"a_star"` | `"integer_program"` | `"toulbar2"` | `"variable_elimination"` |
| --- | --- | --- | --- | --- |
| Oracle (no `nodes`) | HMM only | | | |
| Oracle (with `nodes`) | | | | BN, MN, DBN |
| Pyomo | | all models | | |
| Toulbar2 | | | all models | |
| Algebraic | | all models | all models | |

### When to use which

- **Oracle** — most flexible (any Python logic). Without `nodes`, used with
  A\* for HMMs. With `nodes`, used with variable elimination for BN/MN/DBN.
- **Pyomo** — natural for users familiar with Pyomo. Requires a MIP solver.
- **Toulbar2** — targets the Toulbar2 solver. Requires `pytoulbar2`.
- **Algebraic** — the most portable linear constraint format. Write once, run
  with either `"integer_program"` or `"toulbar2"`. Requires `smoek`.